<a href="https://colab.research.google.com/github/Umang-Raval/DSSE_Assignment1/blob/Week2_LightWeight/Assignment1_Group5_Week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Install and import the required dependencies

**You may add or remove based on your assigned model!**

In [ ]:
!pip install -U tokenizers transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# 1. Authentication & Model selection

**Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**CHANGE THIS TO YOUR ASSIGNED MODEL**

In [ ]:
# A smaller model that fits on a free Colab GPU
model_name = "ByteDance-Seed/Seed-Coder-8B-Instruct"

# 2. Hardware optimization (Quantization)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 3. Load the tokenizer & model

In [ ]:
print(f"Loading Tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=False # Does your model need it?
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # What about right?

Loading Tokenizer for ByteDance-Seed/Seed-Coder-8B-Instruct...


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

In [ ]:
print("Loading Model on Colab T4 GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True, # Does your model need it?
    quantization_config=bnb_config, # from section 2 above
    torch_dtype=torch.float16
)

Loading Model on Colab T4 GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

# 4. INFERENCE & HYPERPARAMETER TUNING

**Design the prompt (Does this design/technique have a name?)**

In [ ]:
prompt = [
    {
        "role": "system",
        "content": "You are a helpful software assistant. Your job is to explain the functionality of the provided code in simple terms."
    },
    {
        "role": "user",
        "content": """Please analyze the following source code:

        <source_code>
        public class Task {

            public static void main(String[] args) {
                String id1   = "AMQ-2104";
                double files = 8.0;
                String id2   = "AMQ-317";
                double dmm   = 0.45;

                int n = Integer.parseInt(id1.split("-")[1])
                      + Integer.parseInt(id2.split("-")[1]);

                int digits = 0;
                while (n > 0) {
                    n = n / 10;
                    digits++;
                }

                int impact = (int) (files * dmm);

                System.out.println("Combined digits: " + digits);
                System.out.println("Impact: " + impact);
            }
        }
        </source_code>"""
    }
]

**Convert the prompt into tokens**

In [ ]:
inputs = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

**Understand these hyperparameters and then tune them for better inference**

In [ ]:
print("Generating response...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=512, # which tokens?
    temperature=0.5, # ?
    top_p=0.8, # ?
    do_sample=True, # What if this toggle is false?
    pad_token_id=tokenizer.eos_token_id
)

Generating response...



**Convert the output back from tokens into human-readable text and display it**

In [ ]:
input_length = inputs['input_ids'].shape[1]
response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print("Model output:")
print(response)

Model output:
Certainly! Let's break down the functionality of this Java code:

1. The code defines a class named `Task` with a `main` method, which is the entry point of the program.

2. It initializes four variables:
   - `id1` and `id2` are strings representing task IDs.
   - `files` is a double representing a file size or quantity.
   - `dmm` is a double representing a decimal multiplier.

3. The code then extracts and adds the numeric parts from `id1` and `id2`:
   - It splits each ID by the "-" character and takes the second part (index 1).
   - These parts are parsed as integers and added together.
   - The result is stored in `n`.

4. It counts the number of digits in `n`:
   - It uses a while loop to divide `n` by 10 repeatedly until it becomes 0.
   - Each division represents removing a digit, so it increments a `digits` counter.

5. It calculates an "impact" value:
   - It multiplies `files` and `dmm` and casts the result to an integer.

6. Finally, it prints two pieces of i